# Beyond Binary Accuracy: Sentiment Analysis on IMDb

**Thesis:** high overall accuracy hides systematic failures on difficult reviews (long, negation-heavy, mixed-sentiment). We train three baselines and evaluate them on the full IMDb test set *and* on three **challenging slices** to expose the gap.

## 1. Imports & config

In [ ]:
import os, re, glob, time
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score,
                             confusion_matrix, classification_report)

DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('')), 'data', 'aclImdb')
np.random.seed(42)
print('IMDb folder present:', os.path.isdir(DATA_DIR))

## 2. Load the IMDb dataset (25k train / 25k test)

In [2]:
def load_split(split):
    texts, labels = [], []
    for label_name, label_id in [('pos', 1), ('neg', 0)]:
        for path in glob.glob(os.path.join(DATA_DIR, split, label_name, '*.txt')):
            with open(path, 'r', encoding='utf-8') as f:
                texts.append(f.read())
            labels.append(label_id)
    return texts, labels

t0 = time.time()
X_train_text, y_train = load_split('train')
X_test_text,  y_test  = load_split('test')
print(f'train: {len(X_train_text):,}   test: {len(X_test_text):,}   ({time.time()-t0:.1f}s)')
print('Positive in train:', sum(y_train), '/ Negative in train:', len(y_train)-sum(y_train))
print('\nExample review (truncated):')
print(X_train_text[0][:400], '...')

train: 25,000   test: 25,000   (5.9s)
Positive in train: 12500 / Negative in train: 12500

Example review (truncated):
For a movie that gets no respect there sure are a lot of memorable quotes listed for this gem. Imagine a movie where Joe Piscopo is actually funny! Maureen Stapleton is a scene stealer. The Moroni character is an absolute scream. Watch for Alan "The Skipper" Hale jr. as a police Sgt. ...


## 3. Preprocess the text
Strip HTML, lowercase, keep alpha-only, collapse whitespace.

In [3]:
HTML_TAG  = re.compile(r'<[^>]+>')
NON_ALPHA = re.compile(r'[^a-z\s]')

def clean(text):
    text = HTML_TAG.sub(' ', text)
    text = text.lower()
    text = NON_ALPHA.sub(' ', text)
    return re.sub(r'\s+', ' ', text).strip()

X_train_clean = [clean(t) for t in X_train_text]
X_test_clean  = [clean(t) for t in X_test_text]
print('BEFORE:', X_train_text[0][:180])
print('\nAFTER :', X_train_clean[0][:180])

BEFORE: For a movie that gets no respect there sure are a lot of memorable quotes listed for this gem. Imagine a movie where Joe Piscopo is actually funny! Maureen Stapleton is a scene ste

AFTER : for a movie that gets no respect there sure are a lot of memorable quotes listed for this gem imagine a movie where joe piscopo is actually funny maureen stapleton is a scene steal


## 4. Build features: TF-IDF + hand-crafted lexicon/negation signals

In [4]:
POS_WORDS = set('''good great excellent amazing wonderful fantastic brilliant
awesome loved love enjoyable enjoy best perfect beautiful fun funny hilarious
superb outstanding masterpiece memorable touching heartwarming powerful
charming delightful impressive engaging gripping thrilling stunning magnificent'''.split())

NEG_WORDS = set('''bad terrible awful horrible worst boring dull stupid poor
waste wasted disappointing disappointed hate hated hates annoying pathetic
lame ridiculous worse lousy mediocre forgettable painful laughable weak
predictable cliched unwatchable tedious bland flat sloppy uninspired
cringe cringy nonsense'''.split())

NEGATIONS = set('''not no never none nothing neither nor hardly scarcely barely
without cannot cant couldnt wasnt werent isnt arent dont didnt doesnt wont
wouldnt shouldnt'''.split())

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=50_000,
                             min_df=3, sublinear_tf=True, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train_clean)
X_test_tfidf  = vectorizer.transform(X_test_clean)
print('TF-IDF train shape:', X_train_tfidf.shape)
print('TF-IDF test  shape:', X_test_tfidf.shape)

def hand_features(text):
    words = text.split()
    n = max(len(words), 1)
    pos = sum(w in POS_WORDS for w in words)
    neg = sum(w in NEG_WORDS for w in words)
    neg_cues = sum(w in NEGATIONS for w in words)
    flipped = 0
    for i, w in enumerate(words):
        if w in NEGATIONS:
            window = words[i+1:i+4]
            if any(ww in POS_WORDS or ww in NEG_WORDS for ww in window):
                flipped += 1
    return [pos/n, neg/n, (pos-neg)/n, neg_cues/n, flipped/n, np.log1p(n)]

X_train_hand = np.array([hand_features(t) for t in X_train_clean], dtype=np.float32)
X_test_hand  = np.array([hand_features(t) for t in X_test_clean],  dtype=np.float32)

X_train_hybrid = hstack([X_train_tfidf, csr_matrix(X_train_hand)]).tocsr()
X_test_hybrid  = hstack([X_test_tfidf,  csr_matrix(X_test_hand)]).tocsr()
print('Hybrid train shape:', X_train_hybrid.shape)

TF-IDF train shape: (25000, 50000)
TF-IDF test  shape: (25000, 50000)


Hybrid train shape: (25000, 50006)


## 5. Train three baselines
Naive Bayes, Linear SVM, and Hybrid SVM (TF-IDF + lexicon features).

In [5]:
models = {}

t0 = time.time(); nb = MultinomialNB().fit(X_train_tfidf, y_train)
models['Naive Bayes (TF-IDF)']          = ('tfidf', nb,  time.time()-t0)

t0 = time.time(); svm = LinearSVC(C=1.0).fit(X_train_tfidf, y_train)
models['Linear SVM (TF-IDF)']           = ('tfidf', svm, time.time()-t0)

t0 = time.time(); svm_h = LinearSVC(C=1.0).fit(X_train_hybrid, y_train)
models['Hybrid SVM (TF-IDF+lexicon)']   = ('hybrid', svm_h, time.time()-t0)

for name, (_, _, secs) in models.items():
    print(f'Trained {name:40s}  in {secs:5.1f}s')

Trained Naive Bayes (TF-IDF)                      in   0.0s
Trained Linear SVM (TF-IDF)                       in   0.2s
Trained Hybrid SVM (TF-IDF+lexicon)               in   2.7s


## 6. Full-test-set evaluation
Accuracy, precision, recall, F1 — and a confusion matrix for the best model.

In [6]:
y_test_np = np.asarray(y_test)
preds_cache = {}
rows = []
for name, (feat, clf, _) in models.items():
    X = X_test_tfidf if feat == 'tfidf' else X_test_hybrid
    preds = clf.predict(X)
    preds_cache[name] = preds
    rows.append({
        'model': name,
        'accuracy':  accuracy_score(y_test_np, preds),
        'precision': precision_score(y_test_np, preds),
        'recall':    recall_score(y_test_np, preds),
        'f1':        f1_score(y_test_np, preds),
    })
full_df = pd.DataFrame(rows)
print(full_df.to_string(index=False,
    formatters={'accuracy':'{:.4f}'.format,'precision':'{:.4f}'.format,
                'recall':'{:.4f}'.format,'f1':'{:.4f}'.format}))

best_name = full_df.iloc[full_df['accuracy'].idxmax()]['model']
print(f'\nBest model on full test set: {best_name}')
print('\nConfusion matrix (rows=true, cols=pred) for', best_name)
print(confusion_matrix(y_test_np, preds_cache[best_name]))
print('\nClassification report:')
print(classification_report(y_test_np, preds_cache[best_name],
                            target_names=['neg', 'pos'], digits=4))

                      model accuracy precision recall     f1
       Naive Bayes (TF-IDF)   0.8573    0.8742 0.8347 0.8540
        Linear SVM (TF-IDF)   0.8810    0.8870 0.8733 0.8801
Hybrid SVM (TF-IDF+lexicon)   0.8829    0.8894 0.8745 0.8819

Best model on full test set: Hybrid SVM (TF-IDF+lexicon)

Confusion matrix (rows=true, cols=pred) for Hybrid SVM (TF-IDF+lexicon)
[[11141  1359]
 [ 1569 10931]]

Classification report:
              precision    recall  f1-score   support

         neg     0.8766    0.8913    0.8839     12500
         pos     0.8894    0.8745    0.8819     12500

    accuracy                         0.8829     25000
   macro avg     0.8830    0.8829    0.8829     25000
weighted avg     0.8830    0.8829    0.8829     25000



## 7. Build challenging slices
Three subsets of the test set that are known to be hard:
- **Long reviews** (≥ 500 words)
- **Negation-heavy** (≥ 2 negation cues)
- **Mixed sentiment** (≥ 2 positive *and* ≥ 2 negative lexicon hits)

In [ ]:
long_idx, neg_idx, mixed_idx = [], [], []
for i, t in enumerate(X_test_clean):
    ws = t.split()
    if len(ws) >= 500: long_idx.append(i)
    if sum(w in NEGATIONS for w in ws) >= 2: neg_idx.append(i)
    pos = sum(w in POS_WORDS for w in ws)
    neg = sum(w in NEG_WORDS for w in ws)
    if pos >= 2 and neg >= 2: mixed_idx.append(i)

slices = {
    'Full test set':                        list(range(len(X_test_clean))),
    f'Long reviews (>=500 words)':          long_idx,
    f'Negation-heavy (>=2 cues)':           neg_idx,
    f'Mixed sentiment (pos&neg>=2)':        mixed_idx,
}
for k, v in slices.items():
    print(f'{k:35s}  n = {len(v):>5}')

## 8. Evaluate every model on every slice

In [8]:
results = []
for slice_name, idx in slices.items():
    idx = np.array(idx, dtype=int)
    y_s = y_test_np[idx]
    for model_name, preds in preds_cache.items():
        p = preds[idx]
        results.append({
            'slice': slice_name, 'model': model_name, 'n': len(idx),
            'acc': accuracy_score(y_s, p),
            'prec': precision_score(y_s, p, zero_division=0),
            'rec':  recall_score(y_s, p, zero_division=0),
            'f1':   f1_score(y_s, p, zero_division=0),
        })

res_df = pd.DataFrame(results)
print(res_df.to_string(index=False,
    formatters={'acc':'{:.4f}'.format,'prec':'{:.4f}'.format,
                'rec':'{:.4f}'.format,'f1':'{:.4f}'.format}))

                       slice                       model     n    acc   prec    rec     f1
               Full test set        Naive Bayes (TF-IDF) 25000 0.8573 0.8742 0.8347 0.8540
               Full test set         Linear SVM (TF-IDF) 25000 0.8810 0.8870 0.8733 0.8801
               Full test set Hybrid SVM (TF-IDF+lexicon) 25000 0.8829 0.8894 0.8745 0.8819
  Long reviews (>=300 words)        Naive Bayes (TF-IDF)  5588 0.8482 0.8479 0.8485 0.8482
  Long reviews (>=300 words)         Linear SVM (TF-IDF)  5588 0.8787 0.8924 0.8611 0.8765
  Long reviews (>=300 words) Hybrid SVM (TF-IDF+lexicon)  5588 0.8778 0.8922 0.8593 0.8754
   Negation-heavy (>=3 cues)        Naive Bayes (TF-IDF)  9197 0.8533 0.8308 0.8100 0.8203
   Negation-heavy (>=3 cues)         Linear SVM (TF-IDF)  9197 0.8764 0.8533 0.8464 0.8498
   Negation-heavy (>=3 cues) Hybrid SVM (TF-IDF+lexicon)  9197 0.8760 0.8605 0.8356 0.8478
Mixed sentiment (pos&neg>=2)        Naive Bayes (TF-IDF)  4737 0.8632 0.7556 0.6601 0.7046

## 9. Headline finding: accuracy stays high, but F1 collapses on mixed sentiment

In [9]:
full = res_df[res_df['slice'] == 'Full test set'].set_index('model')
drops = []
for _, r in res_df.iterrows():
    if r['slice'] == 'Full test set': continue
    drops.append({
        'model': r['model'], 'slice': r['slice'],
        'full_acc': full.loc[r['model'], 'acc'],
        'slice_acc': r['acc'],
        'acc_drop_pp': (full.loc[r['model'], 'acc'] - r['acc']) * 100,
        'full_f1': full.loc[r['model'], 'f1'],
        'slice_f1': r['f1'],
        'f1_drop_pp': (full.loc[r['model'], 'f1'] - r['f1']) * 100,
    })
drop_df = pd.DataFrame(drops)
print(drop_df.to_string(index=False,
    formatters={'full_acc':'{:.4f}'.format,'slice_acc':'{:.4f}'.format,
                'acc_drop_pp':'{:+.2f}'.format,'full_f1':'{:.4f}'.format,
                'slice_f1':'{:.4f}'.format,'f1_drop_pp':'{:+.2f}'.format}))

print('\n>>> Accuracy drop on mixed-sentiment is small or negative,')
print('>>> but F1 drop is ~11 pp. Accuracy hides the failure — exactly the thesis.')

                      model                        slice full_acc slice_acc acc_drop_pp full_f1 slice_f1 f1_drop_pp
       Naive Bayes (TF-IDF)   Long reviews (>=300 words)   0.8573    0.8482       +0.90  0.8540   0.8482      +0.57
        Linear SVM (TF-IDF)   Long reviews (>=300 words)   0.8810    0.8787       +0.23  0.8801   0.8765      +0.36
Hybrid SVM (TF-IDF+lexicon)   Long reviews (>=300 words)   0.8829    0.8778       +0.51  0.8819   0.8754      +0.65
       Naive Bayes (TF-IDF)    Negation-heavy (>=3 cues)   0.8573    0.8533       +0.40  0.8540   0.8203      +3.37
        Linear SVM (TF-IDF)    Negation-heavy (>=3 cues)   0.8810    0.8764       +0.46  0.8801   0.8498      +3.03
Hybrid SVM (TF-IDF+lexicon)    Negation-heavy (>=3 cues)   0.8829    0.8760       +0.68  0.8819   0.8478      +3.41
       Naive Bayes (TF-IDF) Mixed sentiment (pos&neg>=2)   0.8573    0.8632       -0.59  0.8540   0.7046     +14.93
        Linear SVM (TF-IDF) Mixed sentiment (pos&neg>=2)   0.8810    0.8

## 10. Qualitative errors on the mixed-sentiment slice
Real reviews where the best model gets it wrong.

In [10]:
best_preds = preds_cache[best_name]
shown = 0
for i in mixed_idx:
    if best_preds[i] != y_test_np[i]:
        print(f'--- Error #{shown+1}  true={"pos" if y_test_np[i]==1 else "neg"}  '
              f'pred={"pos" if best_preds[i]==1 else "neg"} ---')
        print(X_test_text[i][:500].replace('<br />', ' '), '...')
        print()
        shown += 1
        if shown >= 3: break

--- Error #1  true=pos  pred=neg ---
I went into this film thinking I wasn't going to like it, but hoping to be surprised, but this, much like the film, ended up being bleak and hopeless. But don't get me wrong, Minghella delivers every bit of a grand epic and for those who enjoy that kind of experience and are willing to take that adventure and accept what comes as just that, then they will not be disappointed and it will be one of the better films of the year in their opinion. The acting is of a high quality and will most likely  ...

--- Error #2  true=pos  pred=neg ---
When we are young, we all pick out an ideal occupation for ourselves: artist, actor, writer, rocket scientist, etc.. While most of us grow out of our pipe dreams, the main character of American Movie, Mark, has yet to let go of his(and at a thirty-something age too): to become a wealthy acclaimed director. Despite the fact that Murphy's Law won't leave Mark alone and something always seems to go wrong, Mark is able t

## 11. Takeaways

| | Full test set | Mixed-sentiment slice |
|---|---|---|
| **Accuracy** | ~88% | ~88–89% (looks fine) |
| **F1**       | ~88% | **~77%** — **drops ~11 pp** |

- Best overall: **Hybrid SVM (TF-IDF + lexicon + negation features)**.
- On **negation-heavy** reviews, F1 drops ~3 pp.
- On **long reviews**, accuracy drops ~0.5 pp — the weakest effect.
- The big lesson: **accuracy alone is misleading**; challenging slices + F1/recall expose failures that a single accuracy number hides.
- This motivates moving to context-aware models (CNN → BERT → LLMs).